In [3]:
import pandas as pd
import numpy as np
from scipy import stats

# 1. Load the data
# Mapping file provides which model belongs to which option for each question
mapping_df = pd.read_csv('Summary Rating Mapping.csv')
# Survey file contains the actual 1-5 responses
survey_df = pd.read_csv('Summary Evaluation Study - Rating Scale.csv')

# 2. Setup configuration
models_list = ['Fact', 'Hierarchical', 'NLI', 'Original']
num_questions = 20

def clean_rating(val):
    """Extracts the integer 5 from '5 - Best'"""
    if pd.isna(val): return None
    try:
        return int(str(val).split(' ')[0])
    except:
        return None

# 3. Process and Map responses
processed_data = []

for q_idx in range(num_questions):
    q_num = q_idx + 1
    # Get the model mapping for this specific question
    m_row = mapping_df[mapping_df['Question Number'] == q_num].iloc[0]
    model_map = {
        'A': m_row['Option A Model'],
        'B': m_row['Option B Model'],
        'C': m_row['Option C Model'],
        'D': m_row['Option D Model']
    }
    
    # Identify the column names in the survey for this question
    # Google Forms labels them: Rate Option A, Rate Option A.1, Rate Option A.2...
    suffix = f".{q_idx}" if q_idx > 0 else ""
    
    for _, response in survey_df.iterrows():
        for letter in ['A', 'B', 'C', 'D']:
            col_name = f"Rate Option {letter}{suffix}"
            model_name = model_map[letter]
            rating_score = clean_rating(response[col_name])
            
            if rating_score is not None:
                processed_data.append({
                    'User': response['Name'],
                    'Question': q_num,
                    'Model': model_name,
                    'Rating': rating_score
                })

# Create a clean DataFrame
analysis_df = pd.DataFrame(processed_data)

# 4. Statistical Analysis
print("--- Form 2: 1 to 5 Rating Analysis ---")

# Grouped results
summary = analysis_df.groupby('Model')['Rating'].agg(['mean', 'median', 'std', 'count'])
print("\nDescriptive Statistics:")
print(summary)

# Prepare lists for Kruskal-Wallis Test
data_arrays = [analysis_df[analysis_df['Model'] == m]['Rating'] for m in models_list]

# Kruskal-Wallis H-test
h_stat, p_val = stats.kruskal(*data_arrays)

print(f"\nKruskal-Wallis H-Statistic: {h_stat:.4f}")
print(f"P-Value: {p_val:.4e}")

if p_val < 0.05:
    print("Result: Significant difference! The models do not perform equally.")
else:
    print("Result: No significant difference in median scores across the models.")

# Optional: Save processed data for further use
analysis_df.to_csv('Processed_Rating_Scores.csv', index=False)

--- Form 2: 1 to 5 Rating Analysis ---

Descriptive Statistics:
                  mean  median       std  count
Model                                          
Fact          4.300000     4.0  0.497451     60
Hierarchical  4.800000     5.0  0.403376     60
NLI           4.616667     5.0  0.584885     60
Original      4.500000     4.5  0.504219     60

Kruskal-Wallis H-Statistic: 30.5619
P-Value: 1.0512e-06
Result: Significant difference! The models do not perform equally.
